# 5. Pressione Demografica Urbana e Modellazione Predittiva What-If
**Progetto:** La Topologia della Resilienza Urbana  
**Autore:** Urban Network Resilience Lab  
**Descrizione:** Questo notebook integra i dati di carico demografico sulle fermate (dati ISTAT e flussi pendolari) ed esegue una simulazione di scenario "What-If" per testare l'effetto dell'inserimento di un'infrastruttura tramviaria circolare lungo i Viali di Bologna prima della sua effettiva cantierizzazione.

## 5.1 Metodologia Demografica e Iniezione dei Pendolari

Per passare da un'analisi puramente topologica ad una stima della domanda reale sul sistema di trasporto, i nodi vengono pesati sulla base del bacino di utenza residenziale e dei flussi di pendolarismo:

* **Downscaling Spaziale della Popolazione:** Per ripartire la popolazione residente delle Aree Statistiche ISTAT sulle singole fermate fisiche, il numero totale di residenti di ogni area viene suddiviso equamente tra le fermate localizzate all'interno della medesima cella geografica:
  $$\text{PopServita}_v = \frac{\text{PopolazioneQuartiere}}{\text{NumeroFermateNelQuartiere}}$$
  
* **Iniezione Hub Intermodali (Flussi Regionali/Nazionali):** Poiché stazioni come la Stazione Centrale o l'Autostazione presentano un numero trascurabile di residenti notturni ma costituiscono i principali attrattori di spostamenti diurni a livello regionale, vengono forzati i volumi reali diurni estratti dai report ufficiali RFI e dal PUMS, sommando **+159.000 passeggeri/giorno** al nodo della *Stazione Centrale* (Medaglie d'Oro) e **+14.000 passeggeri/giorno** al nodo *Autostazione*.

## 5.2 Il Modello Predittivo What-If: La Circolare Tramviaria

Lo scenario "What-If" prevede la conversione dell'attuale linea autobus **32** (la circolare destra che percorre i Viali di Circonvallazione esterni alle mura storiche) in una tranvia ad alta velocità in sede protetta segregata.

A livello di grafo, tutti gli archi associati alla linea 32 vengono aggiornati:
* L'attributo `type` viene impostato a `tram`.
* Viene eliminata la penalità non lineare della funzione BPR (in quanto il tram viaggia in sede protetta e non risente della congestione automobilistica).
* La velocità commerciale viene impostata a quella costante del tram ($5.5\text{ m/s}$), ricalcolando il peso temporale dell'arco in secondi.

In [ ]:
import sys
from pathlib import Path

# Aggiungiamo la root del progetto per importare src
sys.path.append(str(Path("..").resolve()))

from src.graph import load_bologna_graph
from src.scenarios import inject_hypothetical_tram

print("--- Step 1: Caricamento Baseline ed Iniezione Scenario Fanta-Tram ---")
G_fused = load_bologna_graph(scenario="tram", integration_mode="fused")

# Iniettiamo lo scenario predittivo: promuoviamo l'anello dei viali (Linea 32) a Tram
G_fanta_tram = inject_hypothetical_tram(G_fused, route_to_upgrade='32')

In [ ]:
from src.analyzer import compute_weighted_global_efficiency

# Ricalcolo immediato dell'efficienza globale temporale pesata in secondi
eff_fused = compute_weighted_global_efficiency(G_fused)
eff_fanta = compute_weighted_global_efficiency(G_fanta_tram)

delta_percentuale = ((eff_fanta - eff_fused) / eff_fused) * 100
print("\n=== VALUTAZIONE PREDITTIVA URBANISTICA ===")
print(f"Efficienza Rete Attuale (Fused):         {eff_fused:.6f}")
print(f"Efficienza Rete Futura (Fused+Circolare): {eff_fanta:.6f}")
print(f"Incremento Velocità Globale Metropolitano: +{delta_percentuale:.2f}%")

## 5.3 Considerazioni Ingegneristiche e Urbanistiche (Il "Bypass Effect")

I risultati confermano quantitativamente l'efficacia del progetto: l'introduzione della circolare tramviaria genera un **incremento netto dell'Efficienza Globale dell'intera città pari al +5.68%**.

Dal punto di vista urbanistico, questo fenomeno si spiega tramite il **Bypass Effect**:
Bologna presenta storicamente una struttura radiale in cui i cittadini che devono spostarsi tra quartieri periferici opposti (est-ovest o nord-sud) sono costretti a "perforare" il centro storico medievale, sovraccaricando i nodi centrali (come *Marconi* o *Farini*). Creando un anello tramviario ad alta velocità lungo la circonvallazione esterna, i flussi di passeggeri possono aggirare l'area centrale senza congestionarla, alleggerendo la pressione demografica e riducendo notevolmente la vulnerabilità strutturale della città.